In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
!pip install transformers
!pip install nltk
!pip install scikit-learn


In [ ]:
import pandas as pd

news = pd.read_csv("news.csv")
stock = pd.read_csv("stock.csv")

print(news.head())
print(stock.head())

                                           Headlines  \
0  Jim Cramer: A better way to invest in the Covi...   
1     Cramer's lightning round: I would own Teradyne   
2                                                NaN   
3  Cramer's week ahead: Big week for earnings, ev...   
4  IQ Capital CEO Keith Bliss says tech and healt...   

                             Time  \
0   7:51  PM ET Fri, 17 July 2020   
1   7:33  PM ET Fri, 17 July 2020   
2                             NaN   
3   7:25  PM ET Fri, 17 July 2020   
4   4:24  PM ET Fri, 17 July 2020   

                                         Description  
0  "Mad Money" host Jim Cramer recommended buying...  
1  "Mad Money" host Jim Cramer rings the lightnin...  
2                                                NaN  
3  "We'll pay more for the earnings of the non-Co...  
4  Keith Bliss, IQ Capital CEO, joins "Closing Be...  
         Date   Open   High    Low  Close  Adj Close    Volume
0  01-07-2010  5.000  5.184  4.054  4.392      

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Remove NaN rows first
news = news.dropna(subset=['Headlines'])

def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = text.lower()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

news['clean_text'] = news['Headlines'].apply(clean_text)

print(news[['Headlines','clean_text']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...


                                           Headlines  \
0  Jim Cramer: A better way to invest in the Covi...   
1     Cramer's lightning round: I would own Teradyne   
3  Cramer's week ahead: Big week for earnings, ev...   
4  IQ Capital CEO Keith Bliss says tech and healt...   
5  Wall Street delivered the 'kind of pullback I'...   

                                          clean_text  
0  jim cramer better way invest covid vaccine gol...  
1              cramer lightning round would teradyne  
3  cramer week ahead big week earnings even bigge...  
4  iq capital ceo keith bliss says tech healthcar...  
5  wall street delivered kind pullback waiting ji...  


[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Convert Time column to datetime
news['Date'] = pd.to_datetime(news['Time'], errors='coerce').dt.date

# Convert stock Date column
stock['Date'] = pd.to_datetime(stock['Date'], format='%d-%m-%Y').dt.date

print(news[['Time','Date']].head())
print(stock[['Date']].head())

/tmp/ipython-input-477/2072634370.py:2: FutureWarning: Parsed string " 7:51  PM ET Fri, 17 July 2020" included an un-recognized timezone "ET". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  news['Date'] = pd.to_datetime(news['Time'], errors='coerce').dt.date
/tmp/ipython-input-477/2072634370.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  news['Date'] = pd.to_datetime(news['Time'], errors='coerce').dt.date
/tmp/ipython-input-477/2072634370.py:2: FutureWarning: Parsed string " 7:33  PM ET Fri, 17 July 2020" included an un-recognized timezone "ET". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to conv

                             Time        Date
0   7:51  PM ET Fri, 17 July 2020  2020-07-17
1   7:33  PM ET Fri, 17 July 2020  2020-07-17
3   7:25  PM ET Fri, 17 July 2020  2020-07-17
4   4:24  PM ET Fri, 17 July 2020  2020-07-17
5   7:36  PM ET Thu, 16 July 2020  2020-07-16
         Date
0  2010-07-01
1  2010-07-02
2  2010-07-06
3  2010-07-07
4  2010-07-08


/tmp/ipython-input-477/2072634370.py:2: FutureWarning: Parsed string " 7:05  PM ET Mon, 26 Nov 2018" included an un-recognized timezone "ET". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  news['Date'] = pd.to_datetime(news['Time'], errors='coerce').dt.date
/tmp/ipython-input-477/2072634370.py:2: FutureWarning: Parsed string "12:26  PM ET Thu, 29 Nov 2018" included an un-recognized timezone "ET". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  news['Date'] = pd.to_datetime(news['Time'], errors='coerce').dt.date
/tmp/ipython-input-477/2072634370.py:2: FutureWarning: Parsed string " 7:02  PM ET Tue, 20 Nov 2018" included an un-recognized timezone "ET". Dropping unrecognized timezones is deprecated; in a fu

In [ ]:
# Remove ET manually before converting
news['Time'] = news['Time'].str.replace('ET', '', regex=False)

news['Date'] = pd.to_datetime(news['Time'], errors='coerce').dt.date

/tmp/ipython-input-477/3268122605.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  news['Date'] = pd.to_datetime(news['Time'], errors='coerce').dt.date


In [ ]:
print(news[['Time','Date']].head())

                           Time        Date
0   7:51  PM  Fri, 17 July 2020  2020-07-17
1   7:33  PM  Fri, 17 July 2020  2020-07-17
3   7:25  PM  Fri, 17 July 2020  2020-07-17
4   4:24  PM  Fri, 17 July 2020  2020-07-17
5   7:36  PM  Thu, 16 July 2020  2020-07-16


In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

news['sentiment'] = news['clean_text'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

print(news[['clean_text','sentiment']].head())

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


                                          clean_text  sentiment
0  jim cramer better way invest covid vaccine gol...     0.4404
1              cramer lightning round would teradyne     0.0000
3  cramer week ahead big week earnings even bigge...     0.0000
4  iq capital ceo keith bliss says tech healthcar...     0.5719
5  wall street delivered kind pullback waiting ji...     0.5267


In [ ]:
daily_sentiment = news.groupby('Date')['sentiment'].mean().reset_index()

print(daily_sentiment.head())

         Date  sentiment
0  2017-12-22  -0.175267
1  2017-12-26   0.250000
2  2017-12-27   0.000000
3  2018-01-02   0.190486
4  2018-01-03   0.232233


In [ ]:
final_data = stock.merge(daily_sentiment, on='Date')

print(final_data.head())
print("Shape:", final_data.shape)

         Date       Open       High        Low      Close  Adj Close  \
0  2017-12-22  65.902000  66.183998  64.963997  65.040001  65.040001   
1  2017-12-26  64.765999  64.788002  63.316002  63.458000  63.458000   
2  2017-12-27  63.200001  63.535999  62.150002  62.327999  62.327999   
3  2018-01-02  62.400002  64.421997  62.200001  64.106003  64.106003   
4  2018-01-03  64.199997  65.050003  63.110001  63.450001  63.450001   

     Volume  sentiment  
0  21079000  -0.175267  
1  21892000   0.250000  
2  23560500   0.000000  
3  21761000   0.190486  
4  22607500   0.232233  
Shape: (592, 8)


In [ ]:
# Create target variable
final_data['target'] = (final_data['Close'].shift(-1) > final_data['Close']).astype(int)

# Remove last row (because shift creates NaN)
final_data = final_data.dropna()

print(final_data[['Date','Close','sentiment','target']].head())
print("Shape:", final_data.shape)

         Date      Close  sentiment  target
0  2017-12-22  65.040001  -0.175267       0
1  2017-12-26  63.458000   0.250000       0
2  2017-12-27  62.327999   0.000000       1
3  2018-01-02  64.106003   0.190486       0
4  2018-01-03  63.450001   0.232233       0
Shape: (592, 9)


In [ ]:
print(final_data.isnull().sum())

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
sentiment    0
target       0
dtype: int64


In [ ]:
features = final_data[['Close', 'Volume', 'sentiment']]
target = final_data['target']

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

split = int(len(features_scaled) * 0.8)

X_train = features_scaled[:split]
X_test = features_scaled[split:]

y_train = target[:split]
y_test = target[split:]

import numpy as np

X_train = np.reshape(X_train, (X_train.shape[0], 1, X_train.shape[1]))
X_test = np.reshape(X_test, (X_test.shape[0], 1, X_test.shape[1]))

print(X_train.shape)

(473, 1, 3)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential()
model.add(LSTM(64, input_shape=(1, 3)))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

history = model.fit(X_train, y_train,
                    epochs=15,
                    batch_size=16,
                    validation_data=(X_test, y_test))

Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.4487 - loss: 0.6935 - val_accuracy: 0.5378 - val_loss: 0.6924
Epoch 2/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4754 - loss: 0.6933 - val_accuracy: 0.5462 - val_loss: 0.6926
Epoch 3/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4689 - loss: 0.6933 - val_accuracy: 0.5630 - val_loss: 0.6915
Epoch 4/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5062 - loss: 0.6932 - val_accuracy: 0.5630 - val_loss: 0.6908
Epoch 5/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5213 - loss: 0.6928 - val_accuracy: 0.5630 - val_loss: 0.6909
Epoch 6/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4998 - loss: 0.6933 - val_accuracy: 0.5630 - val_loss: 0.6903
Epoch 7/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5078 - loss: 0.6932 - val_accuracy: 0.5630 - val_loss: 0.6904
Epoch 8/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5153 - loss: 0.6930 - val_accuracy: 0.5630 - val_loss: 0.690

In [ ]:
from sklearn.metrics import classification_report

# Predictions
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

print(classification_report(y_test, y_pred))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        52
           1       0.56      1.00      0.72        67

    accuracy                           0.56       119
   macro avg       0.28      0.50      0.36       119
weighted avg       0.32      0.56      0.41       119



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import numpy as np

sequence_length = 5

X = []
y = []

for i in range(sequence_length, len(features_scaled)):
    X.append(features_scaled[i-sequence_length:i])
    y.append(target.iloc[i])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (587, 5, 3)
y shape: (587,)


In [ ]:
split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)

(469, 5, 3)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(5, 3)))
model.add(Dropout(0.2))
model.add(LSTM(32))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test)
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.4778 - loss: 0.6935 - val_accuracy: 0.5593 - val_loss: 0.6905
Epoch 2/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5315 - loss: 0.6926 - val_accuracy: 0.5593 - val_loss: 0.6902
Epoch 3/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5112 - loss: 0.6930 - val_accuracy: 0.5593 - val_loss: 0.6905
Epoch 4/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5025 - loss: 0.6937 - val_accuracy: 0.5593 - val_loss: 0.6895
Epoch 5/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4826 - loss: 0.6939 - val_accuracy: 0.5593 - val_loss: 0.6907
Epoch 6/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5294 - loss: 0.6925 - val_accuracy: 0.5593 - val_loss: 0.6896
Epoch 7/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5099 - loss: 0.6926 - val_accuracy: 0.5593 - val_loss: 0.6894
Epoch 8/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5146 - loss: 0.6923 - val_accuracy: 0.5593 - val_loss: 0.6903


In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

print(classification_report(y_test, y_pred))

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        52
           1       0.56      1.00      0.72        66

    accuracy                           0.56       118
   macro avg       0.28      0.50      0.36       118
weighted avg       0.31      0.56      0.40       118



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# reshape 3D to 2D
X_train_rf = X_train.reshape(X_train.shape[0], -1)
X_test_rf = X_test.reshape(X_test.shape[0], -1)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_rf, y_train)

y_pred_rf = rf.predict(X_test_rf)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.52      0.33      0.40        52
           1       0.59      0.76      0.66        66

    accuracy                           0.57       118
   macro avg       0.55      0.54      0.53       118
weighted avg       0.56      0.57      0.55       118



In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

rf.fit(X_train_rf, y_train)

y_pred_rf = rf.predict(X_test_rf)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.40      0.19      0.26        52
           1       0.55      0.77      0.64        66

    accuracy                           0.52       118
   macro avg       0.47      0.48      0.45       118
weighted avg       0.48      0.52      0.47       118



In [ ]:
stock['Return'] = stock['Close'].pct_change()
stock['MA_5'] = stock['Close'].rolling(5).mean()

In [ ]:
stock.dropna(inplace=True)

In [ ]:
stock.head()
stock.isnull().sum()

,0
Date,0
Open,0
High,0
Low,0
Close,0
Adj Close,0
Volume,0
Return,0
MA_5,0


In [ ]:
print(stock.shape)

(2839, 9)
